# Dependências

In [2]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
#!pip install gcloud
#!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling



c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [3]:
diretorio = 'G:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\415 - Repositório de Dados\\Repositório Local\\PNAD'

In [4]:
os.chdir(diretorio)  

In [6]:
os.listdir(diretorio)

['2025', '2024', 'pnad_indicadores_serie_2016_2025.xlsx']

In [12]:
df = pd.read_excel('pnad_indicadores_serie_2016_2025.xlsx', sheet_name='indicador_07')
df

,ano,sexo,cor,freq,freq_se,freq_cv,prop,prop_se,prop_cv
0,2016,Homem,Branca,150918.139490,10656.621934,0.070612,0.626129,0.023250,0.037132
1,2016,Homem,Negra,88488.282113,6581.831728,0.074381,0.367120,0.023747,0.064683
2,2016,Homem,Outra,1627.043379,1003.422568,0.616715,0.006750,0.004173,0.618160
3,2016,Mulher,Branca,96823.103556,6999.153516,0.072288,0.635953,0.027328,0.042971
4,2016,Mulher,Negra,54586.277753,5276.553342,0.096664,0.358533,0.027473,0.076626
5,2016,Mulher,Outra,839.384713,601.358131,0.716427,0.005513,0.003947,0.715850
6,2017,Homem,Branca,135176.129989,9617.558389,0.071148,0.612900,0.026614,0.043422
7,2017,Homem,Negra,83407.190296,7279.403718,0.087275,0.378175,0.026250,0.069411
8,2017,Homem,Outra,1968.226236,938.558342,0.476855,0.008924,0.004279,0.479442
9,2017,Mulher,Branca,92900.173183,8048.158848,0.086632,0.654076,0.028633,0.043776


In [10]:
df1 = pd.read_excel('G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\PNAD\\2024\\pnad_indicadores_output_v2.xlsx', sheet_name='Dados7')
df1

,cor,sexo,sum,prop,cv
0,Branca,Homem,106974.783587,58.8,5.539487
1,Branca,Mulher,71083.821453,59.1,8.336432
2,Negra,Homem,72186.303832,39.7,6.251243
3,Negra,Mulher,47657.997900,39.6,9.468591
4,Outra,Homem,2625.781575,1.4,33.100077
5,Outra,Mulher,1504.452713,1.3,57.770890


In [11]:
df1['ano'] = 2024
df1["sum"] = df1["sum"].round().astype("Int64")
df1 = df1.drop(columns=['cv'])
df1 = df1.rename(columns= {'sum':'quantidade_vinculos', 'cor':'cor_raca','sexo':'genero'}) 
df1 = df1[['ano', 'cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df1


,ano,cor_raca,genero,quantidade_vinculos,prop
0,2024,Branca,Homem,106975,58.8
1,2024,Branca,Mulher,71084,59.1
2,2024,Negra,Homem,72186,39.7
3,2024,Negra,Mulher,47658,39.6
4,2024,Outra,Homem,2626,1.4
5,2024,Outra,Mulher,1504,1.3


In [16]:
df = (
    df.groupby(["ano", "genero", "cor_raca"], as_index=False)
       .agg({"quantidade_vinculos": "sum"})
)

# calcular total por ano
df["total_ano"] = (
    df.groupby("ano")["quantidade_vinculos"].transform("sum")
)

# calcular proporção correta
df["prop_genero_cor_ano"] = (
    df["quantidade_vinculos"] / df["total_ano"]
) * 100

df.head()

,ano,genero,cor_raca,quantidade_vinculos,total_ano,prop_genero_cor_ano
0,2025,Homem,Branca,1145904,3372556,33.977316
1,2025,Homem,Negra,832824,3372556,24.694149
2,2025,Homem,Outra,38980,3372556,1.1558
3,2025,Mulher,Branca,850997,3372556,25.232998
4,2025,Mulher,Negra,486384,3372556,14.421821


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ano                  6 non-null      int64  
 1   genero               6 non-null      object 
 2   cor_raca             6 non-null      object 
 3   quantidade_vinculos  6 non-null      Int64  
 4   total_ano            6 non-null      Int64  
 5   prop_genero_cor_ano  6 non-null      Float64
dtypes: Float64(1), Int64(2), int64(1), object(2)
memory usage: 438.0+ bytes


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ano                  6 non-null      int64  
 1   genero               6 non-null      object 
 2   cor_raca             6 non-null      object 
 3   quantidade_vinculos  6 non-null      Int64  
 4   total_ano            6 non-null      Int64  
 5   prop_genero_cor_ano  6 non-null      Float64
dtypes: Float64(1), Int64(2), int64(1), object(2)
memory usage: 438.0+ bytes


In [14]:
df_merged= pd.merge(df, df1, on=['ano', 'cor_raca', 'genero', 'quantidade_vinculos', 'prop'], how='outer')
df_merged

,ano,cor_raca,genero,quantidade_vinculos,prop
0,2024,Branca,Homem,106975,58.80
1,2024,Branca,Mulher,71084,59.10
2,2024,Negra,Homem,72186,39.70
3,2024,Negra,Mulher,47658,39.60
4,2024,Outra,Homem,2626,1.40
5,2024,Outra,Mulher,1504,1.30
6,2025,Branca,Homem,117232,34.15
7,2025,Branca,Mulher,81821,23.83
8,2025,Negra,Homem,79384,23.12
9,2025,Negra,Mulher,52770,15.37


# Upload

In [15]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [17]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('cor_raca', 'STRING', description= 'Raça/cor autodeclarado ou não'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('cargos_lideranca')

table_ref = dataset_ref.table('PNAD_vinculos_lideranca_genero_cor') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df_merged, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=7fbe4e0f-6c07-48f1-ab1b-d55d2efa67fe>